# coerce-float-arg-to-array — faded example 3: bias_grad: coerce scalar bias, then unbroadcast gradient to scalar shape

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `coerce-float-arg-to-array`. The last cell reports your progress on the `Backprop: Coerce float arg to array` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: Coerce float arg to array` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`coerce-float-arg-to-array`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "coerce-float-arg-to-array"
DD_SUBTOPIC = "Backprop: Coerce float arg to array"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

For `out = x + b` where `b` is a Python `float` broadcast against a matrix `x`, the backward pass for `b` must sum the incoming gradient over every broadcast axis. Coercing `b` to a 0-D array first means the unbroadcast routine can read `b.shape == ()` and sum the gradient down to a scalar, instead of choking on a bare Python float.

## Faded exercise 3

Implement `bias_grad_b(grad_out, b)`. First coerce `b` to an ndarray if it is a raw scalar, then return the gradient w.r.t. `b`, which (since `d(x+b)/db = 1` and `b` is a scalar broadcast over all of `x`) is the sum of `grad_out` over ALL axes, reshaped to `b`'s shape. Complete the blanked coercion line.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
import numpy as np

def bias_grad_b(grad_out, b):
    if not isinstance(b, np.ndarray):
        b = np.array(float(b))  # coerce raw scalar -> 0-D ndarray
    grad_b = grad_out.sum().reshape(b.shape)  # sum over all broadcast axes -> scalar
    return grad_b

np.random.seed(0)
grad_out = np.random.randn(2, 3)
gb = bias_grad_b(grad_out, 0.5)
print(gb.shape, np.round(float(gb), 4), np.allclose(gb, grad_out.sum()))


def _test():
    import numpy as np
    np.random.seed(0)
    x = np.random.randn(2, 3)
    b = 0.5
    grad_out = np.ones((2, 3))
    gb = bias_grad_b(grad_out, b)
    assert gb.shape == (), "gradient w.r.t. a scalar bias must be 0-D"
    # Independent ground truth via torch autograd.
    xt = t.tensor(x, requires_grad=True)
    bt = t.tensor(float(b), requires_grad=True)
    (xt + bt).sum().backward()
    assert np.allclose(float(gb), float(bt.grad)), "must match torch grad for the bias"
    # General check with non-uniform upstream grad.
    go2 = np.array([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
    gb2 = bias_grad_b(go2, 2.0)
    assert np.isclose(float(gb2), go2.sum())


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import numpy as np

def bias_grad_b(grad_out, b):
    if not isinstance(b, np.ndarray):
        b = np.array(float(b))  # coerce raw scalar -> 0-D ndarray
    grad_b = grad_out.sum().reshape(b.shape)  # sum over all broadcast axes -> scalar
    return grad_b

np.random.seed(0)
grad_out = np.random.randn(2, 3)
gb = bias_grad_b(grad_out, 0.5)
print(gb.shape, np.round(float(gb), 4), np.allclose(gb, grad_out.sum()))
```
</details>